# Day 6 — Hybrid RAG: BM25 + embeddings + RRF + rerank, measured with recall@k
**Theory:** [day-06-rag-for-agents.md](../theory/day-06-rag-for-agents.md) · ⏱ ~75 min · Level: medium

Everything except the final answer runs **free on CPU**. `fastembed` downloads two small ONNX models
(~70–130 MB) the first time.

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone + install the course repo, read the key from notebook secrets
    if not pathlib.Path("agentic-ai-course").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Rahulverma5/agentic-ai-course"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "agentic-ai-course/requirements.txt"], check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
    except Exception as e:
        print("Add GEMINI_API_KEY to the notebook secrets:", e)
    REPO = pathlib.Path("agentic-ai-course").resolve()
else:  # Codespaces / local: find the repo root
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
from google.genai import types  # Gemini SDK types: Content, Part, GenerateContentConfig, ...
from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of, thinking, user
from shopease.data import setup_data
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

## 1. Knowledge base: policy sections + extra FAQ entries

In [ ]:
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt
from rank_bm25 import BM25Okapi

docs = []
for path in sorted(pathlib.Path("projects/shopease-support-agent/data/policies").glob("*.md")):
    for section in re.split(r"\n(?=## )", path.read_text()):
        if section.startswith("## "):
            docs.append({"id": f"{path.stem}#{section.splitlines()[0][3:]}", "text": section})
FAQ = {
    "faq#gift-wrap": "Gift wrapping costs $3 per item and can be added at checkout.",
    "faq#student": "Students get 10% off with a verified student email.",
    "faq#address": "You can change the delivery address until the order ships.",
    "faq#price-match": "ShopEase does not price-match other stores.",
    "faq#international": "ShopEase ships to the US, UK, Canada and India.",
}
docs += [{"id": k, "text": v} for k, v in FAQ.items()]
print(len(docs), "chunks"); pd.DataFrame(docs).head(8)

## 2. A labelled test set (question → the chunk that answers it)
Retrieval needs its own eval. Measure it **before** blaming the LLM.

In [ ]:
TESTS = [
    ("How many days do I have to send something back?", "returns#Return window"),
    ("Will I get my money back on the card I paid with?", "returns#Return window"),
    ("Is return postage free for gold customers?", "returns#Return shipping"),
    ("My headphones are opened, can I still return them?", "returns#Electronics"),
    ("Who approves a big refund?", "returns#Refund approvals"),
    ("When will express delivery arrive?", "shipping#Delivery times"),
    ("My parcel never showed up after two weeks", "shipping#Late or lost orders"),
    ("Can I cancel before it ships?", "shipping#Order changes"),
    ("How long are watches covered against defects?", "warranty#Coverage"),
    ("What do I need to file a defect claim?", "warranty#How to claim"),
    ("Can I pay with PayPal?", "payments#Accepted methods"),
    ("Support asked for my CVV, is that normal?", "payments#Security"),
    ("Do you wrap presents?", "faq#gift-wrap"),
    ("Discount for college students?", "faq#student"),
    ("Do you deliver to India?", "faq#international"),
]

## 3. BM25 (keywords)

In [ ]:
tok = lambda t: re.findall(r"[a-z0-9]+", t.lower())
bm25 = BM25Okapi([tok(d["text"]) for d in docs])
def bm25_rank(q, k=10):
    s = bm25.get_scores(tok(q)); return [docs[i]["id"] for i in np.argsort(-s)[:k]]
print(bm25_rank("parcel never arrived", 3))

## 4. Embeddings (meaning) with fastembed + cosine similarity

In [ ]:
from fastembed import TextEmbedding
try:
    embedder = TextEmbedding("BAAI/bge-small-en-v1.5")   # ~130 MB download the first time
    E = np.array(list(embedder.embed([d["text"] for d in docs])))
    E = E / np.linalg.norm(E, axis=1, keepdims=True)
    def vec_rank(q, k=10):
        qv = np.array(list(embedder.embed([q])))[0]; qv /= np.linalg.norm(qv)
        return [docs[i]["id"] for i in np.argsort(-(E @ qv))[:k]]
    print("embedding dims:", E.shape[1])
    print(vec_rank("parcel never arrived", 3))
except Exception as e:  # no internet / Hugging Face blocked: keep going with BM25 so the rest still runs
    print(f"⚠️ Could not load the embedding model ({type(e).__name__}). Falling back to BM25 for 'vector' results.")
    vec_rank = bm25_rank

## 5. Reciprocal Rank Fusion: combine rankings without calibrating scores

In [ ]:
def rrf(*rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, d in enumerate(ranking, 1):
            scores[d] = scores.get(d, 0) + 1 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)
hybrid_rank = lambda q, k=10: rrf(bm25_rank(q), vec_rank(q))[:k]
print(hybrid_rank("parcel never arrived", 3))

## 6. Rerank with a cross-encoder (scores each question–chunk pair)

In [ ]:
from fastembed.rerank.cross_encoder import TextCrossEncoder
text_of_id = {d["id"]: d["text"] for d in docs}
try:
    reranker = TextCrossEncoder("Xenova/ms-marco-MiniLM-L-6-v2")   # ~90 MB download the first time
    def rerank(q, k=10):
        cands = hybrid_rank(q, 10)
        scores = list(reranker.rerank(q, [text_of_id[c] for c in cands]))
        return [c for _, c in sorted(zip(scores, cands), key=lambda p: -p[0])][:k]
except Exception as e:
    print(f"⚠️ Could not load the reranker ({type(e).__name__}). 'rerank' will return the hybrid ranking.")
    rerank = hybrid_rank

## 7. Measure recall@3 for each method

In [ ]:
def recall_at(fn, k=3):
    return np.mean([gold in fn(q)[:k] for q, gold in TESTS])
scores = {name: recall_at(fn) for name, fn in [("BM25", bm25_rank), ("vector", vec_rank),
                                               ("hybrid (RRF)", hybrid_rank), ("hybrid + rerank", rerank)]}
print(scores)
plt.figure(figsize=(6, 3)); plt.bar(scores.keys(), scores.values()); plt.ylim(0, 1.05); plt.title("recall@3"); plt.show()

In [ ]:
# Which questions does each method miss? (error analysis)
for q, gold in TESTS:
    misses = [n for n, fn in [("bm25", bm25_rank), ("vec", vec_rank), ("hybrid", hybrid_rank)] if gold not in fn(q)[:3]]
    if misses:
        print(f"{q:<55} missed by {misses}")

## 8. Generation with citations (the only paid step)

In [ ]:
def answer(q):
    hits = rerank(q, 3)
    ctx = "\n\n".join(f'<passage source="{h}">\n{text_of_id[h]}\n</passage>' for h in hits)
    return ask(f"{ctx}\n\nAnswer using only the passages and cite sources like (returns#Return window).\nQ: {q}",
               max_tokens=300)
print(answer("My parcel never showed up after two weeks, what can I do?"))

## 9. Agentic search: retrieval as a *tool*
The agent can search, read, rephrase and search again: better for multi-hop questions, more expensive.

In [ ]:
from course_utils import run_agent
SEARCH = {"name": "search_policies", "description": "Search ShopEase policies/FAQ. Returns top passages with sources. "
          "Rephrase and search again if results look weak.",
          "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}
search = lambda query: "\n\n".join(f"[{h}] {text_of_id[h]}" for h in rerank(query, 3))
res = run_agent("I'm a student in India: can I get a discount AND free delivery on a $40 order?",
                [SEARCH], {"search_policies": search}, thinking_level="low")

## ✏️ Exercises
1. Add 5 tricky test questions with exact tokens (e.g. "SKU", "CVV"). Which method wins on those?
2. Try **contextual retrieval**: prefix each chunk with a 1-sentence summary from `ask()` before embedding. Does recall@3 change?
3. Compare with the capstone's version: `from shopease import rag; rag.search("parcel never arrived", mode="hybrid")`.

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker` (estimated at paid prices; the free tier costs $0). Make a habit of checking tokens and cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None